# Danh gia 2-Stage Pipeline (Test Evaluate)

Notebook nay chi thuc hien **DANH GIA** (khong train), yeu cau:
- Da upload model weights len Kaggle Dataset
- Da chuan bi du lieu test

### Cac buoc:
1. Cai dat
2. Chuan bi du lieu test
3. Danh gia pipeline 2-Stage_FINAL (evaluate_2stage.py, conf=0.40)
4. Danh gia pipeline 2-Stage_SAHI (evaluate_2stage.py --sahi, conf=0.35)
5. Zip ket qua

In [ ]:
# ============================================================
# 1. Tai Ma nguon & Cai dat thu vien
# ============================================================
!git clone https://github.com/Shiba-dotcom/waste-detection2-Stage.git
!pip install -q sahi ultralytics timm


In [ ]:
# ============================================================
# 2. Nap Du lieu Ngoai lai (TACO, TrashNet, RealWaste)
# ============================================================
import os, shutil

!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/TrashNet
!mkdir -p /kaggle/working/waste-detection2-Stage/data/external/RealWaste
!mkdir -p /kaggle/working/waste-detection2-Stage/data/raw

datasets_to_copy = [
    {"src": "/kaggle/input/datasets/feyzazkefe/trashnet/dataset-resized",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/TrashNet"},
    {"src": "/kaggle/input/datasets/sohamchaudhari2004/taco-trash-detection-dataset/data",
     "dst": "/kaggle/working/waste-detection2-Stage/data/raw"},
    {"src": "/kaggle/input/datasets/joebeachcapital/realwaste/realwaste-main/RealWaste",
     "dst": "/kaggle/working/waste-detection2-Stage/data/external/RealWaste"}
]

for task in datasets_to_copy:
    if os.path.exists(task["src"]):
        os.makedirs(task["dst"], exist_ok=True)
        shutil.copytree(task["src"], task["dst"], dirs_exist_ok=True)
        print(f"Da tai: {os.path.basename(task['src'])}")
    else:
        print(f"Bo qua: {task['src']} (Khong tim thay tren Kaggle Dataset)")

In [ ]:
# ============================================================
# 3. Chuan bi du lieu test 
# ============================================================
%cd /kaggle/working/waste-detection2-Stage

print("--- Tao nhan YOLO & Split dataset ---")
!python src/data_prep/data_cleaning.py
!python src/Training_dataYolo.py
!python src/data_prep/split_dataset.py

print("\nHoan tat chuan bi du lieu test!")


In [ ]:
# ============================================================
# 4. Danh gia Toan Trinh - Ban 2-Stage_FINAL
#    Model    : models/final_best.pt + models/final_best.pth
#    conf     : 0.40 (nguong toi uu)
#    Macro F1 : tinh tren 5 lop rac (khong tinh Background)
# ============================================================

%cd /kaggle/working/waste-detection2-Stage

!python src/evaluate_2stage.py \
    --detector models/2-Stage_best.pt \
    --classifier models/2-Stage_best.pth \
    --data-dir data/processed/images/test \
    --label-dir data/processed/labels/test \
    --conf 0.4 \
    --output results/eval_final_v2


In [ ]:
# ============================================================
# 5. Danh gia Toan Trinh - Ban 2-Stage_SAHI
#    Model    : stage1_tiled_best.pt + models/final_best.pth
#    conf     : 0.35 (SAHI can nguong thap hon de giu recall)
#    SAHI     : slice=512, overlap=0.2
#    Macro F1 : tinh tren 5 lop rac (khong tinh Background)
# ============================================================

%cd /kaggle/working/waste-detection2-Stage

!python src/evaluate_2stage.py \
    --detector models/2-Stage_SAHI.pt \
    --classifier models/2-Stage_SAHI.pth \
    --data-dir data/processed/images/test \
    --label-dir data/processed/labels/test \
    --conf 0.35 \
    --sahi \
    --output results/eval_sahi_v2


In [ ]:
# ============================================================
# 6. Zip ket qua de tai ve
# ============================================================
import shutil

shutil.make_archive("/kaggle/working/eval_results", "zip",
                    "/kaggle/working/waste-detection2-Stage/results")
print("Da nen ket qua: /kaggle/working/eval_results.zip")
